# Carlos AI no Colab: conversa, imagem e vídeo

Usa modelos abertos prontos, na GPU A100. Os arquivos ficam em `Meu Drive/Carlos-AI`.

**Este notebook nunca foi executado.** Rode a parte 1 e depois só a parte que quiser: use **uma IA por vez** e, para trocar, reinicie a sessão em **Ambiente de execução → Reiniciar sessão** e rode a parte 1 de novo. Confira a licença na página de cada modelo. Ao terminar, desconecte o ambiente para não gastar unidades.

## 1. Preparar (sempre primeiro)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/Carlos-AI'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U diffusers transformers accelerate bitsandbytes sentencepiece protobuf ftfy imageio imageio-ffmpeg

import time
import torch
PASTA = '/content/drive/MyDrive/Carlos-AI/'

## 2. Conversa (Qwen 2.5 72B, em 4 bits)

Troque o texto em `pergunta` e rode a segunda célula quantas vezes quiser. Confira a licença do modelo. Se a primeira célula mostrar partes fora da GPU, as respostas ficam mais lentas.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

NOME = 'Qwen/Qwen2.5-72B-Instruct'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(NOME)
modelo = AutoModelForCausalLM.from_pretrained(NOME, quantization_config=quant, device_map='auto')
print('partes fora da GPU:', sum(1 for d in modelo.hf_device_map.values() if d in ('cpu', 'disk')))

In [ ]:
pergunta = 'Explique em português, de forma simples, o que é um sistema operacional.'

mensagens = [
    {'role': 'system', 'content': 'Você é a Carlos AI. Responda em português do Brasil.'},
    {'role': 'user', 'content': pergunta},
]
texto = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
entrada = tok(texto, return_tensors='pt').to(modelo.device)
inicio = time.time()
saida = modelo.generate(**entrada, max_new_tokens=400)
print(tok.decode(saida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True))
print('segundos:', round(time.time() - inicio, 1))

## 2b. Janela de conversa

Abre um chat aqui no notebook. Para abrir em outra aba, troque `share=False` por `share=True`: isso cria um link público temporário, e quem tiver o link consegue usar a sua sessão, então não compartilhe. Para fechar, pare a célula.

In [ ]:
!pip install -q gradio
import gradio as gr

SISTEMA = 'Você é a Carlos AI. Responda em português do Brasil.'

def responder(mensagem, historico):
    conversa = [{'role': 'system', 'content': SISTEMA}]
    conversa += [{'role': m['role'], 'content': m['content']} for m in historico]
    conversa.append({'role': 'user', 'content': mensagem})
    texto = tok.apply_chat_template(conversa, tokenize=False, add_generation_prompt=True)
    entrada = tok(texto, return_tensors='pt').to(modelo.device)
    saida = modelo.generate(**entrada, max_new_tokens=800)
    return tok.decode(saida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True)

gr.ChatInterface(responder, type='messages', title='Carlos AI').launch(share=False, debug=True)

## 3. Imagem (FLUX.1 schnell)

Em inglês, o modelo costuma responder melhor.

In [ ]:
from diffusers import FluxPipeline

img = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-schnell', torch_dtype=torch.bfloat16).to('cuda')

In [ ]:
pedido = 'a glowing violet aurora over a calm mountain lake at night, detailed, cinematic'

inicio = time.time()
imagem = img(pedido, num_inference_steps=4, guidance_scale=0.0, height=1024, width=1024).images[0]
caminho = PASTA + 'imagem-%d.png' % int(time.time())
imagem.save(caminho)
print('segundos:', round(time.time() - inicio, 1), '| salva em', caminho)
imagem

## 4. Vídeo (Wan 2.1, 14B)

É a parte mais pesada: pode levar muitos minutos por vídeo e gastar mais unidades. Comece com este vídeo curto.

In [ ]:
from diffusers import AutoencoderKLWan, WanPipeline

NOME_VIDEO = 'Wan-AI/Wan2.1-T2V-14B-Diffusers'
vae = AutoencoderKLWan.from_pretrained(NOME_VIDEO, subfolder='vae', torch_dtype=torch.float32)
vid = WanPipeline.from_pretrained(NOME_VIDEO, vae=vae, torch_dtype=torch.bfloat16).to('cuda')

In [ ]:
from diffusers.utils import export_to_video

pedido_video = 'a violet aurora moving slowly over a calm mountain lake at night, cinematic'

inicio = time.time()
quadros = vid(prompt=pedido_video, height=480, width=832, num_frames=33, guidance_scale=5.0, num_inference_steps=30).frames[0]
caminho = PASTA + 'video-%d.mp4' % int(time.time())
export_to_video(quadros, caminho, fps=16)
print('segundos:', round(time.time() - inicio, 1), '| salvo em', caminho)